<a href="https://colab.research.google.com/drive/1lyzG0_bkdP9g2zSaJ2j1n50br29c62gE?usp=sharing" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-google-genai chromadb pymupdf ImageHash pytesseract sentence-transformers
!apt-get install -y tesseract-ocr --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61

In [ ]:
# from langchain_chroma import Chroma

#  langchain-chroma to resolve dependency conflicts.
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-proto opentelemetry-exporter-otlp-proto-grpc opentelemetry-semantic-conventions --quiet
!pip install -U langchain-chroma --quiet



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.43.0 which is incompatible.


In [ ]:
# ── Document loading & parsing ──
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
import pytesseract  # OCR fallback —  (+ apt-get install tesseract-ocr on Colab)
from PIL import Image


from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# ── Vector store ──
from langchain_chroma import Chroma
#import chromadb

# ── LLM (generation) ──
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai  # for direct Gemini multimodal calls

# ── RAG chain ──
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


import os
import json
import time
from pathlib import Path

/tmp/ipykernel_736/2637982335.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
import fitz
from PIL import ImageFile
import pickle
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
from google.colab import userdata

api_key1 = userdata.get('Ushasee_gemini')  # secretly loading API key
api_key2 = userdata.get('RTM_ET_hackathon')  # secretly loading API key 2
api_key3 = userdata.get('Pritam_ET_hackathon')
api_key4 = userdata.get('Mayukh_gemini')
api_key5 = userdata.get('RTM_ET_hackathon_2')

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = api_key4 # set api key


In [ ]:
print("Key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))


Key loaded: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Then point at wherever you put it, e.g.:
DATA_ROOT = "/content/drive/MyDrive/ET_hackathon/raw/"

In [ ]:
os.listdir(DATA_ROOT)

['Procedures',
 'Manuals',
 'Incident_Reports',
 'Regulations',
 'Maintenance_logs']

###  OCR-need detector : optical character recoginiton

In [ ]:
import fitz  # PyMuPDF

def needs_ocr(pdf_path, sample_pages=10) -> bool:
    doc = fitz.open(pdf_path)
    pages_to_check = min(sample_pages, len(doc))
    ocr_needy_pg = 0

    for i in range(pages_to_check):
        text = doc[i].get_text()
        char_count = len(text.strip())
        #print(f"page {i+1} : chars : {char_count}")
        if char_count < 50:  # this page alone has almost no extractable text
            ocr_needy_pg += 1

    doc.close()
    return ocr_needy_pg > pages_to_check * 0.5



#### check all pdfs if ocr needed or not

In [ ]:
for folders in os.listdir(DATA_ROOT):
  pdfs = os.listdir(os.path.join(DATA_ROOT, folders))

  for pdf in pdfs:
    if pdf.endswith(".pdf"):
      full_pdf_path = os.path.join(DATA_ROOT, folders, pdf)
      isneed = needs_ocr(full_pdf_path)
      print(f" {folders}/{pdf} : {isneed}")

# most pdfs dont need OCR

 Procedures/startup_shutdown.pdf : False
 Procedures/Pump_Operation.pdf : False
 Procedures/SOP_pump.pdf : False
 Procedures/Lockout_Tagout_SOP.pdf : False
 Procedures/maintenance_SOP_refinary.pdf : False
 Procedures/OSHA_safety_management.pdf : False
 Procedures/oil_refinary_permits2.pdf : False
 Procedures/oil_refinary_permits.pdf : False
 Procedures/A00_Metal_Pump.pdf : False
 Procedures/BKHazarika.pdf : False
 Manuals/OPERATING_MANUAL_for_Centrifugal_pumps.pdf : False
 Manuals/centrfugal_pump_maintanance.pdf : False
 Manuals/Chapter_Six_Centrifugal_Pump_Maintenance.pdf : False
 Manuals/Compressor_operational.pdf : False
 Manuals/Inspection_Manual_Heat_Exchangers.pdf : False
 Manuals/control_valve.pdf : False
 Manuals/ball_valves_IOM.pdf : False
 Manuals/pumphandbook.pdf : False
 Manuals/Compressed_Air_Manual.pdf : False
 Manuals/BEST_PRACTICE_MANUAL_ELECTRIC_MOTORS.pdf : False
 Manuals/apv_pumps_centrifugal_us.pdf : False
 Manuals/Unit_IV_Centrifugal_Pump.pdf : False
 Manuals/Valve

In [ ]:
import hashlib
import imagehash
import json
import os


CAPTION_CACHE_PATH = "/content/drive/MyDrive/ET_hackathon/processed/caption_cache.json"

def load_caption_cache():
    if os.path.exists(CAPTION_CACHE_PATH):
        with open(CAPTION_CACHE_PATH, "r") as f:
            return json.load(f)
    return {}

def save_caption_cache(cache):
    with open(CAPTION_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4, ensure_ascii=False)


def image_hash(pil_image):
    # Stable hash of image content — same image = same hash, regardless of filename/location
    return str(imagehash.phash(pil_image))

CAPTION_CACHE = load_caption_cache()

### **function for checking is the image meaning full!**
##### we should ignore the images like banner , poster, or very small logo images, which is not carring any information

In [ ]:
import numpy as np
from PIL import Image


def is_meaningful_image(pil_image,min_dim=150, max_aspect_ratio=10.0,
                        entropy_threshold=2.0, non_white_threshold=0.03):
    """
    Returns True only for images likely to contain useful technical content.

    Filters out:
    - tiny icons
    - logos
    - decorative separators
    - blank images
    - duplicate images
    """

    w, h = pil_image.width, pil_image.height


    if w < min_dim or h < min_dim:
        return False

    # 2. Extremely long banner/divider images
    aspect_ratio = max(w, h) / min(w, h)

    if aspect_ratio > max_aspect_ratio:
        return False

    # 3. Image entropy (information content)
    try:
        gray = pil_image.convert("L")
    except Exception:
        return False


    if gray.entropy() < entropy_threshold:
        return False

    extrema = gray.getextrema()
    if extrema[1] - extrema[0] < 20:  # very low contrast = likely flat/blank/simple icon
        return False

    arr = np.array(gray)
    non_white_ratio = np.mean(arr < 245)

    if non_white_ratio < non_white_threshold:
        return False

    return True

### this cell handels diagrams and images,by image captioning through gemini

In [ ]:
# from concurrent.futures import ThreadPoolExecutor
# import google.generativeai as genai
# import io
# from PIL import Image

# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
# #vision_model = genai.GenerativeModel("gemini-2.5-flash")
# vision_model = genai.GenerativeModel("gemini-2.5-flash-lite")

# executor = ThreadPoolExecutor(max_workers=2)

In [ ]:
def extract_images_from_page(doc, page, image_list):
    pil_images = []

    for img in image_list:
        try:
          xref = img[0]
          base_image = doc.extract_image(xref)
          image_bytes = base_image["image"]

         # print(base_image["ext"])

          pil_image = Image.open(io.BytesIO(image_bytes))

          pil_image.load() # force load

          if is_meaningful_image(pil_image):
              pil_images.append(pil_image)

        except Exception as e:
            print(f"Skipping corrupted image (xref={img[0]}): {e}")
            continue

    return pil_images


def caption_image_sync(pil_image):

  for attempt in range(3):
    try:
        response = vision_model.generate_content([
            """You are processing engineering manuals for a RAG system.

        Summarize this engineering diagram in 3–5 concise sentences.
        Mention only the equipment, labels, flow direction, and key components to the point. Ignore decorative elements

        If the image is a table, convert it into structured text (but concised).
        Do not hallucinate.""",
            pil_image
        ])

        return response.text

    except Exception as e:
        if "429" in str(e):
          print("Rate limit hit. Sleeping 40 seconds...")
          time.sleep(40)
          continue

        print(e)
        return None

#############################################

def caption_all_images(pil_images):

    global CAPTION_CACHE
    results = [None] * len(pil_images)

    futures = []
    future_info = []

    for i, img in enumerate(pil_images):

        h = image_hash(img)
        if h in CAPTION_CACHE:
            results[i] = CAPTION_CACHE[h]
        else:
            future = executor.submit(caption_image_sync, img)
            futures.append(future)
            future_info.append((i, h))

    for future, (i, h) in zip(futures, future_info):

        caption = future.result()
        if caption is None:
            continue

        results[i] = caption
        CAPTION_CACHE[h] = caption
    save_caption_cache(CAPTION_CACHE)

    return [r for r in results if r is not None]

In [ ]:
EXTRACTED_PAGES_DIR = "/content/drive/MyDrive/ET_hackathon/processed/extracted_pages/"

def pages_cache_path(filename):
    safe_name = filename.replace(".pdf", "").replace(" ", "_")
    return os.path.join(EXTRACTED_PAGES_DIR, f"{safe_name}_pages.pkl")

def save_extracted_pages(filename, pages):
    with open(pages_cache_path(filename), "wb") as f:
        pickle.dump(pages, f)

def load_extracted_pages(filename):
    path = pages_cache_path(filename)

    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return []

### Native text extraction (non-OCR path)

In [ ]:
async def extract_native_text(pdf_path, filename):

    doc = fitz.open(pdf_path)

    # Resume from previous checkpoint
    pages = load_extracted_pages(filename)

    start_page = len(pages)

    if start_page == len(doc):
      print("PDF already fully extracted.")
      doc.close()
      return pages

    if start_page > 0:
        print(f"Resuming from page {start_page + 1}")

    for i in range(start_page, len(doc)):
        page = doc[i]
        text = page.get_text()

        '''this code snippet is for image captioning via gemini model, as it is expensive we skipping this only extracting
         textual datas and for images we manually give pdf to gemini chat and explain the images to describe'''
        # image_list = page.get_images(full=True)

        # if image_list:
        #     pil_images = extract_images_from_page(doc, page, image_list)
        #     if pil_images:

        #         print(f"Page {i+1}/{len(doc)} : captioning {len(pil_images)} images")

        #         captions = caption_all_images(pil_images)
        #         text += "\n\n[Embedded diagrams/images]\n"
        #         for c in captions:
        #             text += f"\n- {c}"

        pages.append(
            {
                "page_number": i + 1,
                "text": text,
            }
        )

        # Save every 10 pages
        if (i + 1) % 20 == 0:
            save_extracted_pages(filename, pages)
            print(f"Checkpoint saved ({i+1} pages).")

        print(f"Page {i+1} Done.")

    # Final save
    save_extracted_pages(filename, pages)
    doc.close()

    return pages

### OCR extraction path (for scanned PDFs)

In [ ]:
import pytesseract
from PIL import Image
import io
import os

def extract_via_ocr(pdf_path, filename):

   # global CAPTION_CACHE

    doc = fitz.open(pdf_path)

    # Resume if partially processed
    pages = load_extracted_pages(filename)

    start_page = len(pages)

    if start_page > 0:
        print(f"Resuming OCR from page {start_page+1}")

    if start_page == len(doc):
        print("OCR PDF already fully extracted.")
        doc.close()
        return pages

    for i in range(start_page, len(doc)):

        page = doc[i]
        print(f"OCR Page {i+1}/{len(doc)}")

        # Render page
        pix = page.get_pixmap(dpi=300)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        # OCR
        text = pytesseract.image_to_string(img)



        '''
         this code snippet is for image captioning via gemini model, as it is expensive we skipping this only extracting
         textual datas and for images we manually generated pdf through normal gemini chat and explain the images to describe
         '''
        # Caption page image
        # img_hash = image_hash(img)

        # if img_hash in CAPTION_CACHE:
        #     caption = CAPTION_CACHE[img_hash]
        # else:
        #     print(f"Captioning page {i+1}...")
        #     caption = caption_image_sync(img)

        #     if caption is not None:
        #         CAPTION_CACHE[img_hash] = caption
        #         save_caption_cache(CAPTION_CACHE)
        #     else:
        #         print("Caption failed.")

        combined_text = text

        # if caption:
        #     combined_text += "\n\n[Visual Content]\n"
        #     combined_text += caption

        pages.append(
            {
                "page_number": i + 1,
                "text": combined_text,
            }
        )

        # Save every 20 pages
        if (i + 1) % 20 == 0:
            save_extracted_pages(filename, pages)
            print(f"Checkpoint saved ({i+1} pages).")

        print(f"OCR Page {i+1} complete.")

    # Final checkpoint
    save_extracted_pages(filename, pages)

    doc.close()

    return pages

### Unified extraction router


In [ ]:
# confirms the routing decision matches what is expected
# for sample file, and pages come back populated.

async def process_pdf(pdf_path):

    filename = os.path.basename(pdf_path)

    if needs_ocr(pdf_path):
        print(f"[{pdf_path}] → routing to OCR")
        return extract_via_ocr(pdf_path, filename)

    else:
        print(f"[{pdf_path}] → routing to native extraction")
        return await extract_native_text(pdf_path, filename)


### Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
    separators=["\n\n", "\n", ". ", " ", ""]
)


# chunking function
def chunk_pages(pages, source_filename, folder_category):
    chunks = []
    src = source_filename
    if src.endswith("_visual.pdf"):
      src = src.replace("_visual.pdf", ".pdf")

    print(f'chunking {len(pages)} pages')
    for page in pages:
        splits = splitter.split_text(page["text"])
        for j, chunk_text in enumerate(splits):
            chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": src,
                    "page": page["page_number"],
                    "folder": folder_category,
                    "chunk_index": j
                }
            })
    return chunks



## **Embedding setup (Google primary, BGE fallback ready)**

 BGE-large is 1024 dimension


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Downloads the embedding model
embedder = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

###  ChromaDB setup (single collection, persisted to Drive)

In [ ]:
import os
import json
import time
import pickle

DATA_ROOT = "/content/drive/MyDrive/ET_hackathon/raw/"
PROGRESS_LOG_PATH = "/content/drive/MyDrive/ET_hackathon/processed/ingestion_progress.json"

os.makedirs(EXTRACTED_PAGES_DIR, exist_ok=True)

EXCLUDED_FOLDERS = {"Maintenance_logs"}  # CSVs — different pipeline, skip here

def get_pdfs_in_folder(data_root, folder_name):
    folder_path = os.path.join(data_root, folder_name)
    pdf_files = []
    for fname in os.listdir(folder_path):
        if fname.lower().endswith('.pdf'):
            pdf_files.append({
                "path": os.path.join(folder_path, fname),
                "folder_category": folder_name,
                "filename": fname
            })
    return pdf_files

In [ ]:
def load_progress():
    if os.path.exists(PROGRESS_LOG_PATH):
        with open(PROGRESS_LOG_PATH, "r") as f:
            return json.load(f)
    return {}

def save_progress(progress):
    with open(PROGRESS_LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(progress, f, indent=4, ensure_ascii=False) # dump to next lines

progress = load_progress()



In [ ]:
from langchain_chroma import Chroma

persist_dir = "/content/drive/MyDrive/ET_hackathon/processed/chroma_db"

chroma_db = Chroma(  # chroma DB vector DB initialized
    collection_name="industrial_kb",
    embedding_function=embedder,
    persist_directory=persist_dir

)

print("ChromaDB collection initialized.")

def make_chunk_id(source_filename, page_number, chunk_index):
    raw = f"{source_filename}_{page_number}_{chunk_index}"
    return hashlib.md5(raw.encode()).hexdigest()

async def ingest_one_pdf(entry):
    fname = entry["filename"]
    path = entry["path"]
    folder = entry["folder_category"]

    if progress.get(fname) == "done":
        print(f"Skipping (already fully ingested): {fname}")
        return

    print(f"\n=== Processing: {fname} ({folder}) ===")
    start = time.time()

    try:
        # Step 1: extraction — check disk cache first, skip Gemini calls if already done
        #pages = load_extracted_pages(fname)
        # if pages is not None:
        #     print(f"  → Loaded cached extraction ({len(pages)} pages) — no API calls needed")
        pages = await process_pdf(path)

        print(f"  → Extracted {len(pages)} pages, cached to disk")

        # if len(pages) > 0:
        #     print(f"Loaded cached extraction ({len(pages)} pages)")
        # else:
        #     pages = await process_pdf(path)
        #     save_extracted_pages(fname, pages)
        #     print(f"  → Extracted {len(pages)} pages, cached to disk")

        # Step 2: chunk + embed + add — cheap, no API quota cost, safe to redo anytime
        chunks = chunk_pages(pages, fname, folder)
        texts = [c["text"] for c in chunks]
        metadatas = [c["metadata"] for c in chunks]
        ids = [
            make_chunk_id(c["metadata"]["source"], c["metadata"]["page"], c["metadata"]["chunk_index"])
            for c in chunks
        ]

        BATCH_SIZE = 1000

        for i in range(0, len(texts), BATCH_SIZE):
            chroma_db.add_texts(
                texts=texts[i:i+BATCH_SIZE],
                metadatas=metadatas[i:i+BATCH_SIZE],
                ids=ids[i:i+BATCH_SIZE]
            )

    #    chroma_db.add_texts(texts=texts, metadatas=metadatas, ids=ids)

        elapsed = time.time() - start
        print(f"✓ {fname}: {len(chunks)} chunks added in {elapsed:.1f}s")

        progress[fname] = "done"
        save_progress(progress)

    except Exception as e:
        print(f"✗ FAILED: {fname} — {e}")
        progress[fname] = f"failed: {e}"
        save_progress(progress)

ChromaDB collection initialized.


In [ ]:
async def ingest_folder(folder_name):
    pdfs = get_pdfs_in_folder(DATA_ROOT, folder_name)
    print(f"\n{'='*50}\nFOLDER: {folder_name} — {len(pdfs)} PDFs\n{'='*50}")

    for entry in pdfs:
        await ingest_one_pdf(entry)

    done = sum(1 for p in pdfs if progress.get(p["filename"]) == "done")
    print(f"\n{folder_name} complete: {done}/{len(pdfs)} done")

In [ ]:
await ingest_folder("Incident_Reports") # done -- day 1 / embbded with API calling gemini image captioning


FOLDER: Incident_Reports — 6 PDFs

=== Processing: Mechanical_shaft_seals_for_pumps.pdf (Incident_Reports) ===
[/content/drive/MyDrive/ET_hackathon/raw/Incident_Reports/Mechanical_shaft_seals_for_pumps.pdf] → routing to native extraction
Page 1: captioning 2 images...
Page 1 Done...
Page 7: captioning 1 images...
Page 7 Done...
Page 12: captioning 1 images...
Page 12 Done...
Page 17: captioning 1 images...
Page 17 Done...
Page 19: captioning 2 images...
Page 19 Done...
Page 22: captioning 1 images...
Page 22 Done...
Page 25: captioning 1 images...
Page 25 Done...
Page 26: captioning 2 images...
Page 26 Done...
Page 27: captioning 2 images...
Page 27 Done...
Page 28: captioning 3 images...
Page 28 Done...
Page 29: captioning 3 images...
Page 29 Done...
Page 30: captioning 2 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 42.155378335s.
Page 30 Done...
Page 31: captioning 1 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 40.903295289s.
Page 31 Done...
Page 35: captioning 1 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 39.775187023s.
Page 35 Done...
Page 42: captioning 1 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 38.461610842s.
Page 42 Done...
Page 45: captioning 1 images...


...

429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 11.334459569s.
Page 1 Done...
Page 24: captioning 1 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 10.489443444s.
Page 24 Done...
Page 43: captioning 1 images...
Page 43 Done...
Page 45: captioning 1 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 43.768241145s.
Page 45 Done...
Page 68: captioning 1 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 42.225176311s.
Page 68 Done...
Page 69: captioning 1 images...


429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 40.980446536s.
Page 69 Done...
  → Extracted 100 pages, cached to disk
✓ lbna26331enn.pdf: 637 chunks added in 94.1s

=== Processing: xvi_paper_54.pdf (Incident_Reports) ===
[/content/drive/MyDrive/ET_hackathon/raw/Incident_Reports/xvi_paper_54.pdf] → routing to native extraction
  → Extracted 9 pages, cached to disk
✓ xvi_paper_54.pdf: 48 chunks added in 2.1s

=== Processing: IOCL_enquiry.pdf (Incident_Reports) ===
[/content/drive/MyDrive/ET_hackathon/raw/Incident_Repor

In [ ]:
await ingest_folder("Incident_Reports")  # done remaining of it -- day 1 / embbded with API calling gemini image captioning


FOLDER: Incident_Reports — 6 PDFs
Skipping (already fully ingested): Mechanical_shaft_seals_for_pumps.pdf
Skipping (already fully ingested): Mechanical_SHAFT_SEAL_FAILURE.pdf
Skipping (already fully ingested): lbna26331enn.pdf
Skipping (already fully ingested): xvi_paper_54.pdf

=== Processing: IOCL_enquiry.pdf (Incident_Reports) ===
[/content/drive/MyDrive/ET_hackathon/raw/Incident_Reports/IOCL_enquiry.pdf] → routing to OCR
OCR Page 1/32
OCR Page 1 complete....
OCR Page 2/32
OCR Page 2 complete....
OCR Page 3/32
OCR Page 3 complete....
OCR Page 4/32
OCR Page 4 complete....
OCR Page 5/32
OCR Page 5 complete....
OCR Page 6/32
OCR Page 6 complete....
OCR Page 7/32
OCR Page 7 complete....
OCR Page 8/32
OCR Page 8 complete....
OCR Page 9/32
OCR Page 9 complete....
OCR Page 10/32
OCR Page 10 complete....
OCR Page 11/32
OCR Page 11 complete....
OCR Page 12/32
OCR Page 12 complete....
OCR Page 13/32
OCR Page 13 complete....
OCR Page 14/32
OCR Page 14 complete....
OCR Page 15/32
OCR Page 15 c

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6967.05ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 12176.80ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 10721.63ms


OCR Page 23 complete....
OCR Page 24/32
captioning 24 page image...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 8663.82ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4696.03ms


OCR Page 24 complete....
OCR Page 25/32
captioning 25 page image...


...

Rate limit hit. Sleeping 40 seconds...
!!caption faled!!
OCR Page 31 complete....
OCR Page 32/32
captioning 32 page image...


Rate limit hit. Sleeping 40 seconds...


Rate limit hit. Sleeping 40 seconds...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 15512.41ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1019.68ms


Rate limit hit. Sleeping 40 seconds...
!!caption faled!!
OCR Page 32 complete....
  → Extracted 32 pages, cached to disk
✓ IOCL_enquiry.pdf: 436 chunks added in 1034.6s

=== Processing: Seal_Failure_Analysis.pdf (Incident_Reports) ===
[/content/drive/MyDrive/ET_hackathon/raw/Incident_Reports/Seal_Failure_Analysis.pdf] → routing to native extraction
  → Extracted 5 pages, cached to disk
✓ Seal_Failure_Analysis.pdf: 26 chunks added in 2.7s

Incident_Reports complete: 6/6 done


In [ ]:
await ingest_folder('Manuals') # done -- day 1


FOLDER: Manuals — 13 PDFs

=== Processing: OPERATING_MANUAL_for_Centrifugal_pumps.pdf (Manuals) ===
[/content/drive/MyDrive/ET_hackathon/raw/Manuals/OPERATING_MANUAL_for_Centrifugal_pumps.pdf] → routing to native extraction
Page 1: captioning 2 images...
Page 1 Done...
  → Extracted 44 pages, cached to disk
✓ OPERATING_MANUAL_for_Centrifugal_pumps.pdf: 288 chunks added in 32.4s

=== Processing: centrfugal_pump_maintanance.pdf (Manuals) ===
[/content/drive/MyDrive/ET_hackathon/raw/Manuals/centrfugal_pump_maintanance.pdf] → routing to native extraction
Page 65: captioning 1 images...
Page 65 Done...
Page 216: captioning 1 images...
Page 216 Done...
  → Extracted 233 pages, cached to disk
✓ centrfugal_pump_maintanance.pdf: 1269 chunks added in 90.8s

=== Processing: Chapter_Six_Centrifugal_Pump_Maintenance.pdf (Manuals) ===
[/content/drive/MyDrive/ET_hackathon/raw/Manuals/Chapter_Six_Centrifugal_Pump_Maintenance.pdf] → routing to native extraction
Page 10: captioning 1 images...
Page 10 

Rate limit hit. Sleeping 40 seconds...
Page 13 Done...
Page 16: captioning 1 images...
Page 16 Done...
Page 18: captioning 1 images...
Page 18 Done...
Page 22: captioning 1 images...
Page 22 Done...
Page 23: captioning 1 images...
Page 23 Done...
Page 25: captioning 2 images...
Page 25 Done...
Page 26: captioning 1 images...
Page 26 Done...
Page 28: captioning 1 images...
Page 28 Done...
Page 33: captioning 1 images...
Page 33 Done...
Page 35: captioning 1 images...
Page 35 Done...
Page 38: captioning 1 images...


Rate limit hit. Sleeping 40 seconds...


Rate limit hit. Sleeping 40 seconds...


Rate limit hit. Sleeping 40 seconds...
Page 38 Done...
Page 43: captioning 1 images...
Page 43 Done...
  → Extracted 44 pages, cached to disk
✓ Chapter_Six_Centrifugal_Pump_Maintenance.pdf: 285 chunks added in 379.0s

=== Processing: Compressor_operational.pdf (Manuals) ===
[/content/drive/MyDrive/ET_hackathon/raw/Manuals/Compressor_operational.pdf] → routing to native extraction
Page 10: captioning 1 images...
Page 10 Done...
Page 11: captioning 2 images...
Page 11 Done...
Page 12: captioning 2 images...
Page 12 Done...
Page 13: captioning 2 images...
Page 13 Done...
Page 16: captioning 1 images...
Page 16 Done...
Page 18: captioning 1 images...
Page 18 Done...
Page 22: captioning 1 images...
Page 22 Done...
Page 23: captioning 1 images...
Page 23 Done...
Page 25: captioning 2 images...
Page 25 Done...
Page 26: captioning 1 images...
Page 26 Done...
Page 28: captioning 1 images...
Page 28 Done...
Page 33: captioning 1 images...
Page 33 Done...
Page 35: captioning 1 images...
Page 35 D

...

Rate limit hit. Sleeping 40 seconds...


Rate limit hit. Sleeping 40 seconds...


Rate limit hit. Sleeping 40 seconds...


Rate limit hit. Sleeping 40 seconds...


Rate limit hit. Sleeping 40 seconds...


In [ ]:
# done -- day 2 [after manually generated image captioning _visual.pdf as we faced the quota limit hit challeng]
await ingest_folder('Manuals')


FOLDER: Manuals — 26 PDFs
Skipping (already fully ingested): OPERATING_MANUAL_for_Centrifugal_pumps.pdf
Skipping (already fully ingested): centrfugal_pump_maintanance.pdf
Skipping (already fully ingested): Chapter_Six_Centrifugal_Pump_Maintenance.pdf
Skipping (already fully ingested): Compressor_operational.pdf
Skipping (already fully ingested): Inspection_Manual_Heat_Exchangers.pdf

=== Processing: control_valve.pdf (Manuals) ===
[/content/drive/MyDrive/ET_hackathon/raw/Manuals/control_valve.pdf] → routing to native extraction
Resuming from page 121
Page 121 Done.
Page 122 Done.
Page 123 Done.
Page 124 Done.
Page 125 Done.
Page 126 Done.
Page 127 Done.
Page 128 Done.
Page 129 Done.
Page 130 Done.
Page 131 Done.
Page 132 Done.
Page 133 Done.
Page 134 Done.
Page 135 Done.
Page 136 Done.
Page 137 Done.
Page 138 Done.
Page 139 Done.
Checkpoint saved (140 pages).
Page 140 Done.
Page 141 Done.
Page 142 Done.
Page 143 Done.
Page 144 Done.
Page 145 Done.
Page 146 Done.
Page 147 Done.
Page 14

In [ ]:
await ingest_folder("Procedures")  # done -- day 3


FOLDER: Procedures — 16 PDFs

=== Processing: startup_shutdown.pdf (Procedures) ===
[/content/drive/MyDrive/ET_hackathon/raw/Procedures/startup_shutdown.pdf] → routing to native extraction
PDF already fully extracted.
  → Extracted 22 pages, cached to disk
chunking 22 pages
✓ startup_shutdown.pdf: 167 chunks added in 23.8s

=== Processing: Pump_Operation.pdf (Procedures) ===
[/content/drive/MyDrive/ET_hackathon/raw/Procedures/Pump_Operation.pdf] → routing to native extraction
PDF already fully extracted.
  → Extracted 13 pages, cached to disk
chunking 13 pages
✓ Pump_Operation.pdf: 62 chunks added in 4.8s

=== Processing: SOP_pump.pdf (Procedures) ===
[/content/drive/MyDrive/ET_hackathon/raw/Procedures/SOP_pump.pdf] → routing to native extraction
PDF already fully extracted.
  → Extracted 24 pages, cached to disk
chunking 24 pages
✓ SOP_pump.pdf: 69 chunks added in 5.0s

=== Processing: Lockout_Tagout_SOP.pdf (Procedures) ===
[/content/drive/MyDrive/ET_hackathon/raw/Procedures/Lockout

In [ ]:
await ingest_folder("Regulations")   # done -- day 3


FOLDER: Regulations — 12 PDFs

=== Processing: FACTORY_ACT_1948.pdf (Regulations) ===
[/content/drive/MyDrive/ET_hackathon/raw/Regulations/FACTORY_ACT_1948.pdf] → routing to native extraction
Page 1 Done.
Page 2 Done.
Page 3 Done.
Page 4 Done.
Page 5 Done.
Page 6 Done.
Page 7 Done.
Page 8 Done.
Page 9 Done.
Page 10 Done.
Page 11 Done.
Page 12 Done.
Page 13 Done.
Page 14 Done.
Page 15 Done.
Page 16 Done.
Page 17 Done.
Page 18 Done.
  → Extracted 18 pages, cached to disk
chunking 18 pages
✓ FACTORY_ACT_1948.pdf: 24 chunks added in 1.6s

=== Processing: DOE_handbook.pdf (Regulations) ===
[/content/drive/MyDrive/ET_hackathon/raw/Regulations/DOE_handbook.pdf] → routing to native extraction
Page 1 Done.
Page 2 Done.
Page 3 Done.
Page 4 Done.
Page 5 Done.
Page 6 Done.
Page 7 Done.
Page 8 Done.
Page 9 Done.
Page 10 Done.
Page 11 Done.
Page 12 Done.
Page 13 Done.
Page 14 Done.
Page 15 Done.
Page 16 Done.
Page 17 Done.
Page 18 Done.
Page 19 Done.
Checkpoint saved (20 pages).
Page 20 Done.
Page 